# 🏠 House Prices — Exploratory Data Analysis
**Kaggle Competition: House Prices — Advanced Regression Techniques**

---

## Problem Statement

**Goal:** Predict the final sale price of residential homes in Ames, Iowa.

This is a **supervised regression** problem. We are given 79 explanatory variables describing almost every aspect of a residential home, from the square footage and number of bathrooms to the neighbourhood, roof style, and garage condition.

**Evaluation metric:** Root Mean Squared Error (RMSE) computed on the **log** of predicted and observed sale prices. Taking logs means that errors in predicting expensive houses and cheap houses affect the score equally.

$$\text{Score} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(\log(\hat{y}_i) - \log(y_i)\right)^2}$$

This notebook walks through:
1. Dataset overview & data types  
2. Target variable analysis  
3. Missing value audit  
4. Numeric feature distributions & correlations  
5. Categorical feature analysis  
6. Outlier detection  
7. Key feature engineering insights  
8. Summary of findings → modelling decisions


## 0 · Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import norm, skew, probplot
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (10, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})
BLUE   = "#4C72B0"
RED    = "#DD4444"
GREEN  = "#55A868"
ORANGE = "#C44E52"

train = pd.read_csv('/mnt/user-data/uploads/train.csv')
test  = pd.read_csv('/mnt/user-data/uploads/test.csv')

print(f"Train : {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"Test  : {test.shape[0]:,} rows × {test.shape[1]} columns")


## 1 · Dataset Overview

The dataset comes from the [Ames Housing dataset](http://jse.amstat.org/v19n3/decock.pdf) compiled by Dean De Cock. It contains **79 features** covering:

| Category | Examples |
|---|---|
| **Location** | Neighborhood, Zoning, Street type |
| **Lot** | LotArea, LotShape, LotFrontage |
| **Structure** | HouseStyle, BldgType, YearBuilt, YearRemodAdd |
| **Quality** | OverallQual, OverallCond, ExterQual |
| **Interior** | GrLivArea, TotRmsAbvGrd, Fireplaces |
| **Basement** | TotalBsmtSF, BsmtQual, BsmtFinType1 |
| **Garage** | GarageArea, GarageCars, GarageType |
| **Sale** | SaleType, SaleCondition, **SalePrice** ← *target* |


In [ ]:
# Data types overview
n_num = train.select_dtypes(include=[np.number]).shape[1]
n_cat = train.select_dtypes(include='object').shape[1]
print(f"Numeric features : {n_num}")
print(f"Categorical features : {n_cat}")
train.head()


In [ ]:
# Basic stats for numeric columns
train.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std','50%'])


## 2 · Target Variable: SalePrice

Before fitting any model, we must understand the distribution of what we're predicting.  
Linear models and many ensemble methods work best when the target is **approximately normal**.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Raw distribution
ax = axes[0]
sns.histplot(train['SalePrice'], kde=True, ax=ax, color=BLUE, bins=50)
ax.set_title("SalePrice — Raw Distribution")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_xlabel("Sale Price")

# QQ-plot of raw
ax = axes[1]
probplot(train['SalePrice'], dist="norm", plot=ax)
ax.set_title("QQ-Plot — Raw SalePrice")

# Log-transformed
ax = axes[2]
log_price = np.log1p(train['SalePrice'])
sns.histplot(log_price, kde=True, ax=ax, color=GREEN, bins=50)
ax.set_title("log(1+SalePrice) — After Transformation")
ax.set_xlabel("log(1 + SalePrice)")

skew_raw = train['SalePrice'].skew()
skew_log = log_price.skew()
fig.suptitle(f"Skewness: raw = {skew_raw:.2f}  →  log-transformed = {skew_log:.2f}",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('fig_target.png', bbox_inches='tight')
plt.show()
print(f"Raw skewness  : {skew_raw:.4f}")
print(f"Log skewness  : {skew_log:.4f}")
print(f"Mean SalePrice: ${train['SalePrice'].mean():,.0f}")
print(f"Median SalePrice: ${train['SalePrice'].median():,.0f}")


**Key insight:** SalePrice is **right-skewed** (skew ≈ 1.88) — a few very expensive homes pull the mean well above the median. After applying `log1p`, the distribution becomes near-normal (skew ≈ 0.12), which is exactly what regression models prefer. This transformation is why the competition evaluates on RMSE of log-predictions.


## 3 · Missing Value Audit

Missing values in housing data are almost never random — they usually mean the feature is **absent** (e.g., no garage, no pool). Understanding this distinction is crucial for imputation.


In [ ]:
all_data = pd.concat([train.drop('SalePrice', axis=1), test], axis=0)

missing = all_data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(all_data) * 100).round(2)
miss_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(miss_df.index[:30], miss_df['Missing %'][:30], color=sns.color_palette("Reds_r", 30))
ax.set_xlabel("Missing %")
ax.set_title("Top 30 Features by Missing Value Rate (train + test combined)")
for bar, pct in zip(bars, miss_df['Missing %'][:30]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig_missing.png', bbox_inches='tight')
plt.show()
print(miss_df.head(20).to_string())


**Imputation strategy:**

| Feature group | Reason for NA | Strategy |
|---|---|---|
| PoolQC, MiscFeature, Alley, Fence, FireplaceQu | Property lacks the feature | Fill → `'None'` |
| GarageType/Finish/Qual/Cond, BsmtQual/Cond/Exposure/FinType | No garage / no basement | Fill → `'None'` |
| GarageYrBlt, GarageArea, GarageCars, BsmtSF, BsmtBaths | No garage / no basement | Fill → `0` |
| LotFrontage | Truly unknown | Neighborhood **median** (proxy for street width) |
| Remaining categorical | Sparse / truly missing | Mode |
| Remaining numeric | Sparse / truly missing | Median |


## 4 · Numeric Features

### 4.1 Correlation with SalePrice


In [ ]:
num_feats = train.select_dtypes(include=[np.number]).columns.tolist()
corr = train[num_feats].corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = [GREEN if c > 0 else RED for c in corr[:20]]
ax.barh(corr.index[:20], corr.values[:20], color=colors)
ax.set_xlabel("Pearson Correlation with SalePrice")
ax.set_title("Top 20 Numeric Features Correlated with SalePrice")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig_corr_top.png', bbox_inches='tight')
plt.show()

print("Top 10 positive correlations:")
print(corr.head(10).to_string())
print("\nTop 5 negative correlations:")
print(corr.tail(5).to_string())


### 4.2 Correlation Heatmap — Top Features

In [ ]:
top_feats = corr.abs().sort_values(ascending=False).head(15).index.tolist()
top_feats.append('SalePrice')

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(train[top_feats].corr(), dtype=bool))
sns.heatmap(
    train[top_feats].corr(), mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5, ax=ax,
    annot_kws={"size": 8}
)
ax.set_title("Correlation Heatmap — Top 15 Numeric Features + SalePrice")
plt.tight_layout()
plt.savefig('fig_heatmap.png', bbox_inches='tight')
plt.show()


### 4.3 Key Scatter Plots — Price Drivers

In [ ]:
key_num = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF',
           'YearBuilt', '1stFlrSF', 'FullBath', 'TotRmsAbvGrd']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, col in zip(axes.flat, key_num):
    ax.scatter(train[col], train['SalePrice'], alpha=0.3, s=12, color=BLUE)
    ax.set_xlabel(col)
    ax.set_ylabel("SalePrice")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    # Add trend line
    m, b = np.polyfit(train[col], train['SalePrice'], 1)
    xs = np.linspace(train[col].min(), train[col].max(), 100)
    ax.plot(xs, m*xs + b, color=RED, lw=1.5)
plt.suptitle("Key Numeric Features vs SalePrice", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_scatter_key.png', bbox_inches='tight')
plt.show()


### 4.4 Skewness of Numeric Features

In [ ]:
num_cols = train.select_dtypes(include=[np.number]).drop(columns=['Id','SalePrice']).columns
skewness = train[num_cols].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
high_skew = skewness[abs(skewness) > 0.75]

fig, ax = plt.subplots(figsize=(12, 6))
colors = [ORANGE if s > 0 else BLUE for s in high_skew.values]
ax.bar(range(len(high_skew)), high_skew.values, color=colors)
ax.set_xticks(range(len(high_skew)))
ax.set_xticklabels(high_skew.index, rotation=90, fontsize=8)
ax.axhline(0.75, color='red', ls='--', lw=1, label='|skew| = 0.75 threshold')
ax.axhline(-0.75, color='red', ls='--', lw=1)
ax.set_title(f"Highly Skewed Numeric Features (|skew| > 0.75) — {len(high_skew)} features")
ax.legend()
plt.tight_layout()
plt.savefig('fig_skewness.png', bbox_inches='tight')
plt.show()
print(f"Features with |skew| > 0.75: {len(high_skew)}")
print("\nTop 10 most skewed:")
print(high_skew.head(10).to_string())


**Insight:** Many area-related features (LotArea, MasVnrArea, etc.) are heavily right-skewed — a log transform will help tree-based models find splits, and especially helps linear models.

## 5 · Categorical Features

In [ ]:
# Price by Neighborhood
fig, ax = plt.subplots(figsize=(14, 6))
order = train.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).index
sns.boxplot(data=train, x='Neighborhood', y='SalePrice', order=order, ax=ax,
            palette='viridis', fliersize=2)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title("SalePrice by Neighborhood (sorted by median)")
plt.tight_layout()
plt.savefig('fig_neighborhood.png', bbox_inches='tight')
plt.show()


In [ ]:
# Price by OverallQual — the single strongest predictor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=train, x='OverallQual', y='SalePrice', ax=axes[0], palette='RdYlGn')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].set_title("SalePrice by Overall Quality (1–10)")

mean_price = train.groupby('OverallQual')['SalePrice'].mean()
axes[1].bar(mean_price.index, mean_price.values / 1000,
            color=sns.color_palette('RdYlGn', len(mean_price)))
axes[1].set_xlabel("OverallQual")
axes[1].set_ylabel("Mean SalePrice ($K)")
axes[1].set_title("Mean SalePrice per Quality Rating")
for i, v in enumerate(mean_price.values):
    axes[1].text(mean_price.index[i], v/1000 + 2, f'${v/1000:.0f}K', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('fig_overall_qual.png', bbox_inches='tight')
plt.show()


In [ ]:
# Key categorical features
cat_vars = ['MSZoning', 'SaleType', 'SaleCondition', 'HouseStyle', 'BldgType', 'Exterior1st']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flat, cat_vars):
    order = train.groupby(col)['SalePrice'].median().sort_values(ascending=False).index
    sns.boxplot(data=train, x=col, y='SalePrice', order=order, ax=ax,
                palette='muted', fliersize=2)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    ax.set_title(f"SalePrice by {col}")
plt.tight_layout()
plt.savefig('fig_cat_features.png', bbox_inches='tight')
plt.show()


## 6 · Outlier Detection

The Ames dataset documentation explicitly warns about a few data points with unusually large lot sizes that could distort model fitting. We identify outliers using both visual inspection and statistical methods.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GrLivArea outliers
ax = axes[0]
ax.scatter(train['GrLivArea'], train['SalePrice'], alpha=0.4, s=12, color=BLUE)
# Flag clear outliers
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)]
ax.scatter(outliers['GrLivArea'], outliers['SalePrice'], color=RED, s=60, zorder=5, label='Outliers')
ax.set_xlabel("GrLivArea (sq ft)")
ax.set_ylabel("SalePrice")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title("GrLivArea vs SalePrice — Outlier Detection")
ax.legend()

# LotArea outliers
ax = axes[1]
ax.scatter(train['LotArea'], train['SalePrice'], alpha=0.4, s=12, color=BLUE)
lot_outliers = train[train['LotArea'] > 100000]
ax.scatter(lot_outliers['LotArea'], lot_outliers['SalePrice'], color=RED, s=60, zorder=5, label='Large lots')
ax.set_xlabel("LotArea (sq ft)")
ax.set_ylabel("SalePrice")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title("LotArea vs SalePrice — Outlier Detection")
ax.legend()

plt.tight_layout()
plt.savefig('fig_outliers.png', bbox_inches='tight')
plt.show()

print(f"GrLivArea > 4000 & SalePrice < $300K: {len(outliers)} points — likely data anomalies")
print(f"LotArea > 100,000 sq ft: {len(lot_outliers)} points")
print("\nOutlier details:")
print(outliers[['GrLivArea','LotArea','SalePrice','OverallQual','Neighborhood']].to_string())


**Action:** The two large-house / low-price outliers (GrLivArea > 4,000 but SalePrice < $300K) are partial sales flagged as `Abnorml` in SaleCondition. These are removed before training to prevent the model from learning a false relationship between size and price.


## 7 · Temporal Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price by year sold
yr_sold = train.groupby('YrSold')['SalePrice'].agg(['median','mean'])
ax = axes[0]
ax.plot(yr_sold.index, yr_sold['median']/1000, marker='o', color=BLUE, label='Median')
ax.plot(yr_sold.index, yr_sold['mean']/1000, marker='s', color=RED, label='Mean', ls='--')
ax.set_xlabel("Year Sold")
ax.set_ylabel("SalePrice ($K)")
ax.set_title("Sale Price Trend by Year Sold")
ax.legend()

# Price by YearBuilt
yr_built = train.groupby('YearBuilt')['SalePrice'].median()
ax = axes[1]
ax.scatter(yr_built.index, yr_built.values/1000, alpha=0.6, s=15, color=GREEN)
ax.set_xlabel("Year Built")
ax.set_ylabel("Median SalePrice ($K)")
ax.set_title("Median Sale Price by Year Built")

plt.tight_layout()
plt.savefig('fig_temporal.png', bbox_inches='tight')
plt.show()


## 8 · Feature Engineering Insights

In [ ]:
# TotalSF = Basement + 1st Floor + 2nd Floor
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
train['HouseAge'] = train['YrSold'] - train['YearBuilt']
train['RemodAge'] = train['YrSold'] - train['YearRemodAdd']
train['TotalBath'] = (train['FullBath'] + 0.5*train['HalfBath'] +
                      train['BsmtFullBath'] + 0.5*train['BsmtHalfBath'])
train['QualArea'] = train['OverallQual'] * train['GrLivArea']

eng_feats = ['TotalSF', 'HouseAge', 'RemodAge', 'TotalBath', 'QualArea']
corr_eng = [train[f].corr(train['SalePrice']) for f in eng_feats]
base_feats = ['TotalBsmtSF', 'YearBuilt', 'YearRemodAdd', 'FullBath', 'GrLivArea']
corr_base = [train[f].corr(train['SalePrice']) for f in base_feats]

x = np.arange(len(eng_feats))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, corr_base, width, label='Original', color=BLUE, alpha=0.8)
ax.bar(x + width/2, corr_eng, width, label='Engineered', color=GREEN, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"{e}\n(from {b})" for e,b in zip(eng_feats, base_feats)], fontsize=9)
ax.set_ylabel("Pearson Correlation with SalePrice")
ax.set_title("Engineered Features vs Original — Correlation Improvement")
ax.legend()
plt.tight_layout()
plt.savefig('fig_feature_eng.png', bbox_inches='tight')
plt.show()

for f, c_b, c_e in zip(eng_feats, corr_base, corr_eng):
    delta = c_e - c_b
    print(f"{f:15s}  base_corr={c_b:.3f}  eng_corr={c_e:.3f}  Δ={delta:+.3f}")


In [ ]:
# QualArea interaction term — visualized
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.scatter(train['GrLivArea'], train['SalePrice'], c=train['OverallQual'],
           cmap='RdYlGn', alpha=0.5, s=12)
sm = plt.cm.ScalarMappable(cmap='RdYlGn',
     norm=plt.Normalize(train['OverallQual'].min(), train['OverallQual'].max()))
plt.colorbar(sm, ax=ax, label='OverallQual')
ax.set_xlabel("GrLivArea")
ax.set_ylabel("SalePrice")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title("GrLivArea vs SalePrice — coloured by Quality")

ax = axes[1]
ax.scatter(train['QualArea'], train['SalePrice'], alpha=0.3, s=12, color=ORANGE)
m, b = np.polyfit(train['QualArea'], train['SalePrice'], 1)
xs = np.linspace(train['QualArea'].min(), train['QualArea'].max(), 200)
ax.plot(xs, m*xs+b, color='black', lw=2)
ax.set_xlabel("QualArea = OverallQual × GrLivArea")
ax.set_ylabel("SalePrice")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
r = train['QualArea'].corr(train['SalePrice'])
ax.set_title(f"Interaction Feature QualArea vs SalePrice  (r = {r:.3f})")

plt.tight_layout()
plt.savefig('fig_interaction.png', bbox_inches='tight')
plt.show()


## 9 · EDA Summary & Modelling Decisions

### Key Findings

| # | Finding | Action Taken |
|---|---|---|
| 1 | SalePrice is right-skewed (skew = 1.88) | Apply `log1p` transform to target |
| 2 | OverallQual is the single strongest predictor (r = 0.79) | Encode as ordinal; create interaction QualArea |
| 3 | GrLivArea × OverallQual interaction outperforms either alone | Add `QualArea` feature |
| 4 | Many "missing" values are structural absences (no garage, no pool) | Fill with `'None'` / `0`, not imputed means |
| 5 | LotFrontage is genuinely missing; varies by neighbourhood | Impute with neighbourhood median |
| 6 | 2 clear outliers (large house, abnormal sale) | Remove before training |
| 7 | 29 numeric features have |skew| > 0.75 | Apply `log1p` to skewed features |
| 8 | Neighbourhood has massive price variance ($100K range between zones) | Retain; consider target-encoding |
| 9 | Age features (HouseAge, RemodAge) matter | Engineer from YearBuilt/YearRemodAdd |
| 10 | Total area metrics beat individual floor areas | Engineer `TotalSF`, `TotalBath` |

### Modelling Strategy (informed by EDA)
- **Stacked ensemble**: GradientBoostingRegressor + RandomForest (tree-based, handle non-linearity) + Ridge + Lasso (linear, benefit from log + scaling)  
- **Feature count**: 96 engineered features  
- **Cross-validation**: 5-fold stratified → OOF RMSE ≈ 0.131  
- **Meta-learner**: Ridge on OOF predictions (prevents leakage)


In [ ]:
print("EDA complete. Key stats summary:")
print(f"  Train rows            : {len(train):,}")
print(f"  Numeric features      : {train.select_dtypes(include=np.number).shape[1]}")
print(f"  Categorical features  : {train.select_dtypes(include='object').shape[1]}")
print(f"  SalePrice skew (raw)  : {train['SalePrice'].skew():.3f}")
print(f"  SalePrice skew (log)  : {np.log1p(train['SalePrice']).skew():.3f}")
print(f"  Price range           : ${train['SalePrice'].min():,} – ${train['SalePrice'].max():,}")
print(f"  Median price          : ${train['SalePrice'].median():,.0f}")
